In [1]:

import os, json, shutil
from pathlib import Path
import torch

from tqdm import tqdm 

# --- SAM3 (native) ---
from sam3.model_builder import build_sam3_video_predictor

# --- Optional, for auxiliary frame IO / overlays later ---
from torchcodec.decoders import VideoDecoder

/home/prince/proj/chicken-behaviour-classifier/sam3/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Run inference

In [2]:
# ---------------------------
# Configuration
# ---------------------------
# VIDEO_PATH = "../data/test.mp4"
VIDEO_PATH = "../data/test_short.mp4"
RESULTS_DIR    = Path("./sam3_results/native_api_run")  # where we persist everything

PROMPTS        = ["chicken", "bird"]                  # one concept per call
START_AT_FRAME = 0
PROP_DIR       = "forward"                            # or "both"/"backward"
MAX_TRACK      = None                                 # None = full video, or an int limit

# Create clean results directory
if RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
(RESULTS_DIR / "per_frame").mkdir(parents=True)
(RESULTS_DIR / "meta").mkdir(parents=True)



In [3]:
# ---------------------------
# (1) Build the SAM3 video predictor
#     We pass gpus_to_use to enable multi-GPU if available.
#     async_loading_frames helps reduce latency during init;
#     video_loader_type stays default ("cv2") in Plan A to avoid patching.
# ---------------------------
gpus = list(range(torch.cuda.device_count())) or []
predictor = build_sam3_video_predictor(
    gpus_to_use=gpus,                  # same as the official notebook
    async_loading_frames=True,         # smaller initial spikes
    video_loader_type="cv2",           # keep upstream default in Plan A
)

INFO 2026-01-16 17:24:35,784 2129915 sam3_video_predictor.py: 300: using the following GPU IDs: [0, 1]
INFO 2026-01-16 17:24:35,960 2129915 sam3_video_predictor.py: 316: 


	*** START loading model on all ranks ***


INFO 2026-01-16 17:24:35,961 2129915 sam3_video_predictor.py: 318: loading model on rank=0 with world_size=2 -- this could take a while ...
INFO 2026-01-16 17:24:41,075 2129915 sam3_video_base.py: 125: setting max_num_objects=10000 and num_obj_for_compile=16
INFO 2026-01-16 17:24:43,704 2129915 sam3_video_predictor.py: 320: loading model on rank=0 with world_size=2 -- DONE locally
INFO 2026-01-16 17:24:43,704 2129915 sam3_video_predictor.py: 377: spawning 1 worker processes
/home/prince/proj/chicken-behaviour-classifier/sam3/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to 

In [4]:
# ---------------------------
# (2) Start a session + record meta
# ---------------------------
resp = predictor.handle_request({
    "type": "start_session",
    "resource_path": VIDEO_PATH,
})
session_id = resp["session_id"]

# Persist minimal run metadata
with open(RESULTS_DIR / "meta" / f"run.json", "w") as f:
    json.dump({
        "video_path": str(VIDEO_PATH),
        "prompts": PROMPTS,
        "session_id": session_id,
        "propagation_direction": PROP_DIR,
    }, f, indent=2)

frame loading (OpenCV) [rank=1]: 100%|██████████| 235/235 [00:00<00:00, 598.16it/s]


In [5]:
# ---------------------------
# (3) Add text prompts (one concept per request)
#     Matches the native API (and your HF code shape).
# ---------------------------
for phrase in PROMPTS:
    resp = predictor.handle_request({
        "type": "add_prompt",
        "session_id": session_id,
        "frame_index": START_AT_FRAME,
        "text": phrase,
    })
    # Optionally persist the first-frame outputs for audit/debug
    torch.save(resp["outputs"], RESULTS_DIR / "per_frame" / f"frame_{START_AT_FRAME:06d}.pt")

# ---------------------------
# (4) Stream propagation (generator) and persist per-frame outputs
#     This is where we keep memory flat: we never accumulate everything.
# ---------------------------
frame_count = 0
for response in predictor.handle_stream_request({
    "type": "propagate_in_video",
    "session_id": session_id,
    "propagation_direction": PROP_DIR,                  # "forward" or "both"
    "start_frame_index": START_AT_FRAME,
    "max_frame_num_to_track": MAX_TRACK,               # None = full video
}):
    fidx   = response["frame_index"]
    out    = response["outputs"]            # dict with masks/ids/boxes/scores (tensors)
    # Persist immediately (zero-copy if tensors already on CPU)
    torch.save(out, RESULTS_DIR / "per_frame" / f"frame_{fidx:06d}.pt")
    frame_count += 1

# ---------------------------
# (5) Close the session when done
# ---------------------------
predictor.handle_request({"type": "close_session", "session_id": session_id})
predictor.shutdown()

print(f"Saved {frame_count} frames into: {RESULTS_DIR}")

[Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
[Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


propagate_in_video:   0%|          | 0/235 [00:00<?, ?it/s]

INFO 2026-01-16 17:25:21,030 2130201 sam3_video_predictor.py: 251: removed session 8151e5cf-a17c-4e22-9821-d7e2ff36cb8f; live sessions: [], GPU memory: 5118 MiB used and 9558 MiB reserved (max over time: 8996 MiB used and 9558 MiB reserved)
INFO 2026-01-16 17:25:21,064 2129915 sam3_video_predictor.py: 251: removed session 8151e5cf-a17c-4e22-9821-d7e2ff36cb8f; live sessions: [], GPU memory: 5118 MiB used and 9948 MiB reserved (max over time: 9334 MiB used and 9948 MiB reserved)
INFO 2026-01-16 17:25:21,065 2129915 sam3_video_predictor.py: 513: shutting down 1 worker processes
INFO 2026-01-16 17:25:21,066 2130201 sam3_video_predictor.py: 485: worker rank=1 shutting down
INFO 2026-01-16 17:25:21,275 2129915 sam3_video_predictor.py: 519: shut down 1 worker processes


Saved 235 frames into: sam3_results/native_api_run


# Reading results

In [3]:
# Reading your saved results later

RESULTS_DIR = Path("./sam3_results/native_api_run")

# meta
meta = json.loads((RESULTS_DIR/"meta"/"run.json").read_text())
print("Prompts:", meta["prompts"], "Video:", meta["video_path"])



FileNotFoundError: [Errno 2] No such file or directory: 'sam3_results/native_api_run/meta/run.json'

In [5]:
# # meta
# meta = json.loads((RESULTS_DIR/"meta"/"run.json").read_text())
# print("Prompts:", meta["prompts"], "Video:", meta["video_path"])

# load any frame
f = 120
out = torch.load(RESULTS_DIR/"per_frame"/f"frame_{f:06d}.pt", map_location="cpu", weights_only=False)
masks      = out.get("masks")        # [N,H,W] bool
object_ids = out.get("object_ids")   # [N]
boxes      = out.get("boxes")        # [N,4] (xywh or xyxy depending on build)
scores     = out.get("scores")       # [N]


FileNotFoundError: [Errno 2] No such file or directory: 'sam3_results/native_api_run/per_frame/frame_000120.pt'

In [ ]:
out